Phase 7 -- Interpretability via GAT attention weights.
 
Loads the trained GAT, extracts per-edge attention coefficients from
layer 1 (operates directly on raw patient features, so it's the most
interpretable layer), and shows which neighboring patients most
influenced specific predictions -- including a global check on
whether learned attention concentrates on same-label neighbors more
than the raw graph structure did.

In [1]:
import importlib.util
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
 
_spec = importlib.util.spec_from_file_location("models05", "05_models.py")
_models05 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_models05)
GAT = _models05.GAT
 
DATA_PATH = "Output/graph_data.pt"
WEIGHTS_PATH = "Output/gat_weights.pt"
CLEAN_CSV = "Data/data_clean.csv"
OUT_DIR = "Output"
 
data = torch.load(DATA_PATH, weights_only=False)
df_clean = pd.read_csv(CLEAN_CSV)  # human-readable version, same row order as data.x
 
gat = GAT(in_channels=data.num_node_features, hidden_channels=16, out_channels=2, heads=8)
gat.load_state_dict(torch.load(WEIGHTS_PATH, weights_only=True))
gat.eval()

Loaded graph: Data(x=[704, 42], edge_index=[2, 10048], y=[704], train_mask=[704], val_mask=[704], test_mask=[704])

MODEL ARCHITECTURES
GCN(
  (conv1): GCNConv(42, 16)
  (conv2): GCNConv(16, 2)
)
GCN trainable parameters: 722

GAT(
  (conv1): GATConv(42, 16, heads=8)
  (conv2): GATConv(128, 2, heads=1)
)
GAT trainable parameters: 6,022

FORWARD PASS SANITY CHECK (untrained weights, just checking shapes)
GCN output shape: torch.Size([704, 2]) (expected [704, 2])
GAT output shape: torch.Size([704, 2]) (expected [704, 2])

Both models produce correctly shaped output. Ready for Phase 6 (training).


GAT(
  (conv1): GATConv(42, 16, heads=8)
  (conv2): GATConv(128, 2, heads=1)
)

Step 1: Forward pass with attention weights.
att1 = (edge_index_with_self_loops, alpha) for layer 1
  alpha shape: [num_edges_incl_self_loops, heads=8]
att2 = same structure for layer 2 (heads=1, already averaged/concat=False)

 GATConv adds self-loops internally by default (add_self_loops=True)
 for the attention computation, even though OUR graph's edge_index
 (from Phase 3/4) has none -- this is a separate internal detail of
 the layer, not a re-introduction of the bug we fixed earlier. It
 means each node also attends to itself, which we'll keep visible
 rather than hide, since "how much does the model rely on the
 patient's own features vs. similar neighbors" is itself informative.

In [2]:
with torch.no_grad():
    out, (att1, att2) = gat(data.x, data.edge_index, return_attention_weights=True)
    probs = torch.softmax(out, dim=1)[:, 1]
    preds = out.argmax(dim=1)
 
edge_index_1, alpha_1 = att1  # layer 1: operates on raw input features
alpha_1_mean = alpha_1.mean(dim=1)  # average across 8 heads -> [num_edges]
 
print(f"Layer-1 attention edge_index shape: {edge_index_1.shape}")
print(f"Layer-1 attention alpha shape (per-head): {alpha_1.shape}")
print(f"Averaged across heads: {alpha_1_mean.shape}")
 
 
def get_top_attended_neighbors(target_node, edge_index, alpha_mean, top_k=5):
    """
    For a given target node, find all edges where it's the destination
    (i.e., messages being aggregated INTO it), and return the top_k
    highest-attention source neighbors.
    """
    # edge_index[1] = destination/target node (message recipient)
    mask = edge_index[1] == target_node
    src_nodes = edge_index[0][mask]
    weights = alpha_mean[mask]
 
    order = torch.argsort(weights, descending=True)
    src_nodes = src_nodes[order][:top_k]
    weights = weights[order][:top_k]
    return src_nodes.tolist(), weights.tolist()
 
 
def describe_patient(idx):
    """Human-readable summary of a patient from the cleaned (pre-encoding) data."""
    row = df_clean.iloc[idx]
    aq_cols = [c for c in df_clean.columns if c.startswith("A") and c.endswith("_Score")]
    aq_total = int(row[aq_cols].sum())
    label = "ASD-positive" if row["Class/ASD"] == 1 else "ASD-negative"
    return (f"age={row['age']:.0f}, gender={'M' if row['gender']==1 else 'F'}, "
            f"AQ-10 total={aq_total}/10, jaundice={'yes' if row['jaundice']==1 else 'no'}, "
            f"family_history={'yes' if row['family_autism_history']==1 else 'no'}, "
            f"ethnicity={row['ethnicity']}, true_label={label}")

Layer-1 attention edge_index shape: torch.Size([2, 10752])
Layer-1 attention alpha shape (per-head): torch.Size([10752, 8])
Averaged across heads: torch.Size([10752])


Step 2: Pick illustrative test-set examples -- a correct positive,
a correct negative, and any misclassified case (most valuable for
a screening-tool interpretability story).

In [3]:
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
y_test = data.y[test_idx]
preds_test = preds[test_idx]
 
correct_pos = test_idx[(y_test == 1) & (preds_test == 1)]
correct_neg = test_idx[(y_test == 0) & (preds_test == 0)]
misclassified = test_idx[y_test != preds_test]
 
examples = {}
if len(correct_pos) > 0:
    examples["Correct ASD-positive prediction"] = correct_pos[0].item()
if len(correct_neg) > 0:
    examples["Correct ASD-negative prediction"] = correct_neg[0].item()
if len(misclassified) > 0:
    examples["Misclassified case"] = misclassified[0].item()
 
print("\n" + "=" * 70)
print("ATTENTION-BASED EXPLANATIONS FOR EXAMPLE PATIENTS")
print("=" * 70)
 
fig, axes = plt.subplots(1, len(examples), figsize=(6 * len(examples), 4.5))
if len(examples) == 1:
    axes = [axes]
 
for ax, (title, node_idx) in zip(axes, examples.items()):
    print(f"\n--- {title} (node {node_idx}) ---")
    print(f"Patient {node_idx}: {describe_patient(node_idx)}")
    print(f"Model prediction: {'ASD-positive' if preds[node_idx]==1 else 'ASD-negative'} "
          f"(P(ASD)={probs[node_idx]:.3f})")
 
    neighbors, weights = get_top_attended_neighbors(node_idx, edge_index_1, alpha_1_mean, top_k=5)
    print("Top attended neighbors (excluding self-loop if present):")
    plot_labels, plot_weights, plot_colors = [], [], []
    for n, w in zip(neighbors, weights):
        tag = "[SELF]" if n == node_idx else f"[node {n}]"
        if n != node_idx:
            print(f"  {tag} attention={w:.3f} | {describe_patient(n)}")
        else:
            print(f"  {tag} attention={w:.3f} (model attending to patient's own features)")
        true_label = df_clean.iloc[n]["Class/ASD"]
        plot_labels.append(f"{'self' if n==node_idx else 'n'+str(n)}\n({'ASD+' if true_label==1 else 'ASD-'})")
        plot_weights.append(w)
        plot_colors.append("#E74C3C" if true_label == 1 else "#3498DB")
 
    ax.barh(plot_labels[::-1], plot_weights[::-1], color=plot_colors[::-1])
    ax.set_xlabel("Attention weight (avg across heads)")
    ax.set_title(title, fontsize=10)
 
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/attention_explanations.png", dpi=120)
plt.close()
print(f"\nSaved attention explanation plots to {OUT_DIR}/attention_explanations.png")


ATTENTION-BASED EXPLANATIONS FOR EXAMPLE PATIENTS

--- Correct ASD-positive prediction (node 38) ---
Patient 38: age=53, gender=F, AQ-10 total=7/10, jaundice=no, family_history=no, ethnicity=White-European, true_label=ASD-positive
Model prediction: ASD-positive (P(ASD)=0.988)
Top attended neighbors (excluding self-loop if present):
  [node 689] attention=0.093 | age=39, gender=F, AQ-10 total=7/10, jaundice=no, family_history=no, ethnicity=White-European, true_label=ASD-positive
  [node 332] attention=0.088 | age=55, gender=M, AQ-10 total=7/10, jaundice=no, family_history=no, ethnicity=White-European, true_label=ASD-positive
  [node 295] attention=0.086 | age=45, gender=F, AQ-10 total=7/10, jaundice=no, family_history=no, ethnicity=White-European, true_label=ASD-positive
  [SELF] attention=0.086 (model attending to patient's own features)
  [node 486] attention=0.086 | age=43, gender=M, AQ-10 total=9/10, jaundice=no, family_history=no, ethnicity=White-European, true_label=ASD-positive


Step 3: Global diagnostic -- does LEARNED attention concentrate on
same-label neighbors MORE than the raw graph structure did?
Compare against the Phase 3 structural homophily (0.884).
 Self-loops excluded from this calculation (they're trivially
 "same-label" since a node always shares its own label).

In [4]:
src, dst = edge_index_1[0], edge_index_1[1]
non_self_mask = src != dst
src_ns, dst_ns, alpha_ns = src[non_self_mask], dst[non_self_mask], alpha_1_mean[non_self_mask]
 
same_label = (data.y[src_ns] == data.y[dst_ns]).float()

 IMPORTANT: comparing raw per-edge averages (same-label edges vs
 different-label edges) is confounded by neighbor-count dilution --
 attention is softmax-normalized PER TARGET NODE, so a node with many
 same-label neighbors necessarily gives each one a smaller slice of
 its attention budget, purely from having more competitors, not
 because the model deprioritizes them semantically. The correct
 comparison is: what SHARE of total attention mass do same-label
 edges capture, versus what share of edges they represent? If
 attention were uniform (unlearned), these two shares would be
 identical by construction.

In [5]:
total_attn_same = alpha_ns[same_label == 1].sum().item()
total_attn_diff = alpha_ns[same_label == 0].sum().item()
frac_attn_same = total_attn_same / (total_attn_same + total_attn_diff)
 
count_same = (same_label == 1).sum().item()
count_diff = (same_label == 0).sum().item()
frac_edges_same = count_same / (count_same + count_diff)
 
print("\n" + "=" * 70)
print("GLOBAL DIAGNOSTIC: learned attention mass vs. structural edge share")
print("=" * 70)
print(f"Same-label edges: {frac_edges_same:.3f} share of all edges "
      f"(this IS the Phase 3 structural homophily, recomputed here)")
print(f"Same-label edges: {frac_attn_same:.3f} share of TOTAL attention mass")
print(f"Difference (attention share - edge share): {frac_attn_same - frac_edges_same:+.3f}")
if abs(frac_attn_same - frac_edges_same) < 0.02:
    print("--> Attention mass roughly tracks the underlying graph structure "
          "proportionally -- it neither strongly amplifies nor undermines "
          "the k-NN homophily. The real interpretability value here is "
          "PER-PATIENT (which specific neighbors mattered for which specific "
          "prediction, as shown above), not a global 'attention beats "
          "structure' claim -- which is an honest and still useful finding.")
elif frac_attn_same > frac_edges_same:
    print("--> Attention mass is MORE concentrated on same-label edges than "
          "raw structure alone -- the learned attention is amplifying the "
          "homophily signal beyond what uniform k-NN aggregation would give.")
else:
    print("--> Attention mass is LESS concentrated on same-label edges than "
          "raw structure -- worth investigating specific misclassified cases "
          "(like the example above) to understand why.")
 
torch.save({"edge_index": edge_index_1, "alpha_mean": alpha_1_mean, "preds": preds, "probs": probs},
           f"{OUT_DIR}/attention_artifacts.pt")
print(f"\nSaved attention artifacts to {OUT_DIR}/attention_artifacts.pt")


GLOBAL DIAGNOSTIC: learned attention mass vs. structural edge share
Same-label edges: 0.885 share of all edges (this IS the Phase 3 structural homophily, recomputed here)
Same-label edges: 0.875 share of TOTAL attention mass
Difference (attention share - edge share): -0.010
--> Attention mass roughly tracks the underlying graph structure proportionally -- it neither strongly amplifies nor undermines the k-NN homophily. The real interpretability value here is PER-PATIENT (which specific neighbors mattered for which specific prediction, as shown above), not a global 'attention beats structure' claim -- which is an honest and still useful finding.

Saved attention artifacts to Output/attention_artifacts.pt
